# R03. Original vs Revised GO Enrichment

This reviewer-response notebook audits the **26 FDR-significant GO terms reported in the original manuscript** against the revised GO enrichment analysis.

## Original manuscript

The original manuscript reported:
- 26 FDR-significant GO terms
- 11 Biological Process (BP)
- 9 Cellular Component (CC)
- 6 Molecular Function (MF)
- Fisher's exact test with BH-FDR correction

The revised analysis uses the **114 exploratory mRNA candidates** generated by the limma reanalysis.

> Important: the revised mRNA analysis has no BH-FDR-significant genes. Therefore, the current GO analysis is explicitly exploratory.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from IPython.display import display

cwd = Path.cwd().resolve()

if cwd.name == "revision":
    PROJECT_ROOT = cwd.parents[1]
elif cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RESULTS_DIR = PROJECT_ROOT / "results"
REVISION_DIR = RESULTS_DIR / "revision"
REVISION_DIR.mkdir(parents=True, exist_ok=True)

GO_FILE = RESULTS_DIR / "GO_enrichment_full_results.csv"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("GO_FILE      =", GO_FILE)
print("REVISION_DIR =", REVISION_DIR)

if not GO_FILE.exists():
    raise FileNotFoundError(
        "GO_enrichment_full_results.csv not found. "
        "Run 04_go_enrichment.ipynb first."
    )


PROJECT_ROOT = /Users/jihopark/Desktop/MCDA_revision_final
GO_FILE      = /Users/jihopark/Desktop/MCDA_revision_final/results/GO_enrichment_full_results.csv
REVISION_DIR = /Users/jihopark/Desktop/MCDA_revision_final/results/revision


## 1. Original manuscript GO terms

In [2]:
ORIGINAL_GO = [('CC', 'mitochondrial outer membrane'), ('MF', 'porin activity'), ('CC', 'pore complex'), ('CC', 'mitochondrial outer membrane translocase complex'), ('MF', 'protein channel activity'), ('BP', 'regulation of cellular respiration'), ('CC', 'mitotic cohesin complex'), ('BP', 'meiotic chromosome segregation'), ('BP', 'centriole-centriole cohesion'), ('CC', 'condensed chromosome, centromeric region'), ('BP', 'attachment of spindle microtubules to kinetochore'), ('CC', 'condensed nuclear chromosome, centromeric region'), ('BP', 'regulation of epithelial cell differentiation involved in kidney development'), ('BP', 'epithelial tube morphogenesis'), ('MF', 'cAMP-dependent protein kinase activity'), ('BP', 'kidney morphogenesis'), ('CC', 'sex chromatin'), ('BP', 'histone ubiquitination'), ('CC', 'PRC1 complex'), ('CC', 'Rab-protein geranylgeranyltransferase complex'), ('MF', 'Rab geranylgeranyltransferase activity'), ('BP', 'protein geranylgeranylation'), ('MF', 'prenyltransferase activity'), ('BP', 'riboflavin transport'), ('MF', 'riboflavin transporter activity'), ('BP', 'riboflavin metabolic process')]

original_go = pd.DataFrame(
    ORIGINAL_GO,
    columns=["Original_Ontology", "Original_GO_term"]
)

print("Original GO terms:", len(original_go))
display(original_go)


Original GO terms: 26


,Original_Ontology,Original_GO_term
0,CC,mitochondrial outer membrane
1,MF,porin activity
2,CC,pore complex
3,CC,mitochondrial outer membrane translocase complex
4,MF,protein channel activity
5,BP,regulation of cellular respiration
6,CC,mitotic cohesin complex
7,BP,meiotic chromosome segregation
8,BP,centriole-centriole cohesion
9,CC,"condensed chromosome, centromeric region"


The list above is transcribed from the 26 labels displayed in the original manuscript's GO enrichment figure. The comparison below uses term names because the manuscript figure did not report all GO IDs in tabular form.


## 2. Load revised GO results

In [3]:
go = pd.read_csv(GO_FILE)

required = {
    "Ontology", "GO_ID", "GO_term",
    "candidate_hits", "term_size",
    "P.Value", "adj.P.Val", "Genes"
}

missing = required - set(go.columns)
if missing:
    raise ValueError(
        f"Revised GO output is missing columns: {sorted(missing)}"
    )

print("Revised GO terms tested:", len(go))
print("Revised BH-FDR < 0.05:", int((go["adj.P.Val"] < 0.05).sum()))

display(
    go[
        [
            "Ontology","GO_ID","GO_term",
            "candidate_hits","P.Value","adj.P.Val","Genes"
        ]
    ].head(15)
)


Revised GO terms tested: 462
Revised BH-FDR < 0.05: 0


,Ontology,GO_ID,GO_term,candidate_hits,P.Value,adj.P.Val,Genes
0,BP,GO:0006335,DNA replication-dependent nucleosome assembly,3,0.000117,0.050183,CHAF1B;HIST1H4B;HIST1H4D
1,CC,GO:0000786,nucleosome,4,0.000217,0.050183,HIST1H2AJ;HIST1H2BL;HIST1H4B;HIST1H4D
2,BP,GO:0035574,histone H4-K20 demethylation,2,0.000953,0.095996,HIST1H4B;HIST1H4D
3,MF,GO:0035575,histone demethylase activity (H4-K20 specific),2,0.000953,0.095996,HIST1H4B;HIST1H4D
4,BP,GO:0045653,negative regulation of megakaryocyte different...,2,0.001229,0.095996,HIST1H4B;HIST1H4D
5,BP,GO:0006336,DNA replication-independent nucleosome assembly,2,0.002886,0.095996,HIST1H4B;HIST1H4D
6,BP,GO:0002731,negative regulation of dendritic cell cytokine...,1,0.003079,0.095996,JAK3
7,BP,GO:0045221,negative regulation of FasL biosynthetic process,1,0.003079,0.095996,JAK3
8,BP,GO:0060562,epithelial tube morphogenesis,1,0.003079,0.095996,PRKX
9,BP,GO:2000696,regulation of epithelial cell differentiation ...,1,0.003079,0.095996,PRKX


## 3. Match original terms to revised GO results

In [4]:
def norm_term(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"[–—−]", "-", x)
    x = re.sub(r"\s+", " ", x)
    return x

original_go["term_key"] = original_go["Original_GO_term"].map(norm_term)
go["term_key"] = go["GO_term"].map(norm_term)

comparison = original_go.merge(
    go[
        [
            "term_key","Ontology","GO_ID","GO_term",
            "candidate_hits","term_size",
            "odds_ratio","P.Value","adj.P.Val",
            "Genes"
        ]
    ],
    on="term_key",
    how="left"
)

comparison["Found_in_revised_GO"] = comparison["GO_ID"].notna()
comparison["Revised_FDR_significant"] = (
    comparison["adj.P.Val"].fillna(1.0) < 0.05
)

comparison["Revised_status"] = np.select(
    [
        comparison["Revised_FDR_significant"],
        comparison["Found_in_revised_GO"] &
        (comparison["P.Value"].fillna(1.0) < 0.05),
        comparison["Found_in_revised_GO"],
    ],
    [
        "FDR-significant",
        "Nominal P<0.05 only",
        "Found but not nominally significant",
    ],
    default="Not represented among tested revised terms"
)

display(
    comparison[
        [
            "Original_Ontology","Original_GO_term",
            "GO_ID","candidate_hits",
            "P.Value","adj.P.Val",
            "Revised_status","Genes"
        ]
    ]
)


,Original_Ontology,Original_GO_term,GO_ID,candidate_hits,P.Value,adj.P.Val,Revised_status,Genes
0,CC,mitochondrial outer membrane,GO:0005741,2,0.078861,0.182826,Found but not nominally significant,CISD1;TOMM40L
1,MF,porin activity,GO:0015288,1,0.018333,0.098489,Nominal P<0.05 only,TOMM40L
2,CC,pore complex,GO:0046930,1,0.027376,0.114978,Nominal P<0.05 only,TOMM40L
3,CC,mitochondrial outer membrane translocase complex,GO:0005742,1,0.030372,0.119929,Nominal P<0.05 only,TOMM40L
4,MF,protein channel activity,GO:0015266,1,0.036336,0.124875,Nominal P<0.05 only,TOMM40L
5,BP,regulation of cellular respiration,GO:0043457,1,0.024371,0.107231,Nominal P<0.05 only,CISD1
6,CC,mitotic cohesin complex,GO:0030892,1,0.006148,0.095996,Nominal P<0.05 only,SGOL1
7,BP,meiotic chromosome segregation,GO:0045132,1,0.012259,0.095996,Nominal P<0.05 only,SGOL1
8,BP,centriole-centriole cohesion,GO:0010457,1,0.018333,0.098489,Nominal P<0.05 only,SGOL1
9,CC,"condensed chromosome, centromeric region",GO:0000779,1,0.021357,0.101719,Nominal P<0.05 only,SGOL1


## 4. R03 summary

In [5]:
summary = pd.DataFrame(
    {
        "Metric": [
            "Original manuscript FDR-significant GO terms",
            "Original terms found in revised GO results",
            "Original terms nominal P < 0.05 in revised analysis",
            "Original terms BH-FDR < 0.05 in revised analysis",
            "All revised GO terms tested",
            "All revised GO terms BH-FDR < 0.05",
        ],
        "Value": [
            len(original_go),
            int(comparison["Found_in_revised_GO"].sum()),
            int(
                (
                    comparison["Found_in_revised_GO"] &
                    (comparison["P.Value"].fillna(1.0) < 0.05)
                ).sum()
            ),
            int(comparison["Revised_FDR_significant"].sum()),
            len(go),
            int((go["adj.P.Val"] < 0.05).sum()),
        ],
    }
)

display(summary)


,Metric,Value
0,Original manuscript FDR-significant GO terms,26
1,Original terms found in revised GO results,26
2,Original terms nominal P < 0.05 in revised ana...,25
3,Original terms BH-FDR < 0.05 in revised analysis,0
4,All revised GO terms tested,462
5,All revised GO terms BH-FDR < 0.05,0


## 5. Original biological themes under the revised analysis

In [6]:
THEME_TERMS = {
    "Mitochondrial": [
        "mitochondrial outer membrane",
        "porin activity",
        "pore complex",
        "mitochondrial outer membrane translocase complex",
        "protein channel activity",
        "regulation of cellular respiration",
    ],
    "Chromosome segregation": [
        "mitotic cohesin complex",
        "meiotic chromosome segregation",
        "centriole-centriole cohesion",
        "condensed chromosome, centromeric region",
        "attachment of spindle microtubules to kinetochore",
        "condensed nuclear chromosome, centromeric region",
    ],
    "Kidney / cAMP": [
        "regulation of epithelial cell differentiation involved in kidney development",
        "epithelial tube morphogenesis",
        "cAMP-dependent protein kinase activity",
        "kidney morphogenesis",
    ],
    "Epigenetic / PRC1": [
        "sex chromatin",
        "histone ubiquitination",
        "PRC1 complex",
    ],
    "Prenylation / Rab": [
        "Rab-protein geranylgeranyltransferase complex",
        "Rab geranylgeranyltransferase activity",
        "protein geranylgeranylation",
        "prenyltransferase activity",
    ],
    "Riboflavin": [
        "riboflavin transport",
        "riboflavin transporter activity",
        "riboflavin metabolic process",
    ],
}

theme_rows = []

for theme, terms in THEME_TERMS.items():
    sub = comparison[
        comparison["Original_GO_term"].isin(terms)
    ]

    theme_rows.append(
        {
            "Theme": theme,
            "Original_terms": len(sub),
            "Found_in_revised": int(sub["Found_in_revised_GO"].sum()),
            "Nominal_P_lt_0.05": int(
                (
                    sub["Found_in_revised_GO"] &
                    (sub["P.Value"].fillna(1.0) < 0.05)
                ).sum()
            ),
            "BH_FDR_lt_0.05": int(
                sub["Revised_FDR_significant"].sum()
            ),
        }
    )

theme_summary = pd.DataFrame(theme_rows)
display(theme_summary)


,Theme,Original_terms,Found_in_revised,Nominal_P_lt_0.05,BH_FDR_lt_0.05
0,Mitochondrial,6,6,5,0
1,Chromosome segregation,6,6,6,0
2,Kidney / cAMP,4,4,4,0
3,Epigenetic / PRC1,3,3,3,0
4,Prenylation / Rab,4,4,4,0
5,Riboflavin,3,3,3,0


## 6. Top revised GO signals for context

In [7]:
top_revised = go[
    [
        "Ontology","GO_ID","GO_term",
        "candidate_hits","term_size",
        "odds_ratio","P.Value","adj.P.Val","Genes"
    ]
].sort_values(
    ["adj.P.Val","P.Value"]
).head(30)

display(top_revised)


,Ontology,GO_ID,GO_term,candidate_hits,term_size,odds_ratio,P.Value,adj.P.Val,Genes
0,BP,GO:0006335,DNA replication-dependent nucleosome assembly,3,31,36.565476,0.000117,0.050183,CHAF1B;HIST1H4B;HIST1H4D
1,CC,GO:0000786,nucleosome,4,96,15.065628,0.000217,0.050183,HIST1H2AJ;HIST1H2BL;HIST1H4B;HIST1H4D
2,BP,GO:0035574,histone H4-K20 demethylation,2,15,51.591608,0.000953,0.095996,HIST1H4B;HIST1H4D
3,MF,GO:0035575,histone demethylase activity (H4-K20 specific),2,15,51.591608,0.000953,0.095996,HIST1H4B;HIST1H4D
4,BP,GO:0045653,negative regulation of megakaryocyte different...,2,17,44.707879,0.001229,0.095996,HIST1H4B;HIST1H4D
5,BP,GO:0006336,DNA replication-independent nucleosome assembly,2,26,27.928788,0.002886,0.095996,HIST1H4B;HIST1H4D
6,BP,GO:0002731,negative regulation of dendritic cell cytokine...,1,1,inf,0.003079,0.095996,JAK3
7,BP,GO:0045221,negative regulation of FasL biosynthetic process,1,1,inf,0.003079,0.095996,JAK3
8,BP,GO:0060562,epithelial tube morphogenesis,1,1,inf,0.003079,0.095996,PRKX
9,BP,GO:2000696,regulation of epithelial cell differentiation ...,1,1,inf,0.003079,0.095996,PRKX


## 7. Export R03 audit results

In [8]:
comparison_out = (
    REVISION_DIR / "R03_original_26_GO_vs_revised.csv"
)

summary_out = (
    REVISION_DIR / "R03_GO_revision_summary.csv"
)

theme_out = (
    REVISION_DIR / "R03_GO_theme_summary.csv"
)

xlsx_out = (
    REVISION_DIR / "R03_original_vs_revised_GO.xlsx"
)

comparison.drop(columns=["term_key"]).to_csv(
    comparison_out,
    index=False
)

summary.to_csv(
    summary_out,
    index=False
)

theme_summary.to_csv(
    theme_out,
    index=False
)

with pd.ExcelWriter(xlsx_out) as writer:
    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )
    comparison.drop(columns=["term_key"]).to_excel(
        writer,
        sheet_name="Original 26 terms",
        index=False
    )
    theme_summary.to_excel(
        writer,
        sheet_name="Theme summary",
        index=False
    )
    top_revised.to_excel(
        writer,
        sheet_name="Top revised GO",
        index=False
    )

print("Saved:")
print(comparison_out)
print(summary_out)
print(theme_out)
print(xlsx_out)


Saved:
/Users/jihopark/Desktop/MCDA_revision_final/results/revision/R03_original_26_GO_vs_revised.csv
/Users/jihopark/Desktop/MCDA_revision_final/results/revision/R03_GO_revision_summary.csv
/Users/jihopark/Desktop/MCDA_revision_final/results/revision/R03_GO_theme_summary.csv
/Users/jihopark/Desktop/MCDA_revision_final/results/revision/R03_original_vs_revised_GO.xlsx


## R03 Interpretation

The original manuscript reported 26 BH-FDR-significant GO terms from the original 23-DEG set. In the revised analysis, GO enrichment is performed on the 114 **exploratory** mRNA candidates because no mRNA survives BH-FDR correction at the differential-expression level.

The revised GO analysis tests 462 GO terms and yields **no BH-FDR-significant enrichment**. Therefore:

- The original claim of significant functional convergence should not be retained as confirmatory evidence.
- Any recurring chromatin, nucleosome, mitochondrial, chromosome-segregation, kidney/cAMP, or epigenetic signals should be described as **exploratory nominal patterns** only.
- Single-gene-supported GO terms should not be interpreted as pathway-level convergence.
- The revised manuscript should clearly distinguish the historical/original GO results from the reanalysis.
